<a href="https://colab.research.google.com/github/DABMASTER-Brought-me-into-this/NeuroLearn/blob/main/SlideMDCNNImageClassifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Got to get a running mean + var

In [ ]:
import numpy as np
import numpy.lib.stride_tricks as lst
import matplotlib.pyplot as plt
import math
import random
import re
import cv2
import csv
from google.colab.patches import cv2_imshow
%matplotlib inline

In [ ]:
# Hyperparameters
START_SIZE = 256
BATCH_SIZE = 6
WIN_SIZE_L1 = (5, 5)
NUM_FEATURES_L1 = 16
MAX_POOLING_SHRINK_L1 = 4
WIN_SIZE_L2 = (5, 5)
NUM_FEATURES_L2 = 64
MAX_POOLING_SHRINK_L2 = 4
WIN_SIZE_L3 = (5, 5)
NUM_FEATURES_L3 = 128
MAX_POOLING_SHRINK_L3 = 2
LEAKY_RELU_CONSTANT = 0.01
EPOCHS = 20
NUM_FEATURES_L4 = 8

In [ ]:
# # Opening the File
# dataset = []
# with open('/content/slidemdimageds.csv', mode='r', newline='') as file:
#     reader = csv.reader(file)
#     for row in reader:
#         if len(row) == 3:
#           dataset.append(row)

In [ ]:
# # Seperating Inputs and Outputs
# last_num_int = lambda a: int(a[-1])
# inputs_seperation = lambda a: a[:2]
# Y = list(map(last_num_int, dataset))
# inputs = list(map(inputs_seperation, dataset))

In [ ]:
# # Preprocessing Text
# stoi = {chr(i): (i - ord('a') + 1) for i in range(ord('a'), ord('z') + 1)}
# stoi['.'] = 0
# itos = {ix: letter for letter, ix in stoi.items()}

In [ ]:
# # Converting All Words Into Numbers
# def tok(word):
#   word = re.sub('[^a-z]', '', word.lower())
#   letters = list(word)
#   enc_letters = list(map(lambda a: stoi[a], letters))
#   if len(enc_letters) < MAX_WORD_LEN:
#     enc_letters.extend([0] * (MAX_WORD_LEN - len(enc_letters)))
#   return enc_letters

In [ ]:
# # Replacing All Text with Tokenized Words
# inputs = [[input[0], tok(input[1])] for input in inputs]

In [ ]:
# Preprocessing Image
images = []
for i in range(0,6):
    img_path = f"image_{i}.jpeg"
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is not None:
        resized_img = cv2.resize(img, (START_SIZE, START_SIZE))
        images.append(resized_img)

X = np.array(images).astype(np.float32)
X /= np.max(X)
Y = np.array([1,0,1,1,1,0])
Y = Y.reshape(-1, 1)

In [ ]:
# PARAPARAMETER!!!
W1 = np.random.randn(*WIN_SIZE_L1, NUM_FEATURES_L1) * 0.01
g1 = np.ones(NUM_FEATURES_L1)
b1 = np.zeros_like(g1)
W2 = np.random.randn(*WIN_SIZE_L2, NUM_FEATURES_L1, NUM_FEATURES_L2) * 0.01
g2 = np.ones(NUM_FEATURES_L2)
b2 = np.zeros_like(g2)
W3 = np.random.randn(*WIN_SIZE_L3, NUM_FEATURES_L2, NUM_FEATURES_L3) * 0.01
g3 = np.ones(NUM_FEATURES_L3)
b3 = np.zeros_like(g3)
W4 = np.random.randn(NUM_FEATURES_L3 * (START_SIZE // MAX_POOLING_SHRINK_L1 // MAX_POOLING_SHRINK_L2 // MAX_POOLING_SHRINK_L3) ** 2, 8) * 0.01
W5 = np.random.randn(NUM_FEATURES_L4, 1) * 0.01

parameters = [W1, g1, b1, W2, g2, b2, W3, g3, b3, W4, W5]

In [ ]:
for epoch in range(EPOCHS):
  for i in range(0, X.shape[0] // BATCH_SIZE, BATCH_SIZE):
    ### Forward Pass
    ## Layer 1
    # Padding
    pad1 = tuple((size-1)//2 for size in WIN_SIZE_L1) # Same Padding Calcs
    img_padded1 = np.pad(X, ((0,0), pad1, pad1), "constant", constant_values=0)

    # Conv
    win_l1 = lst.sliding_window_view(img_padded1, window_shape = WIN_SIZE_L1, axis = (1,2))
    win_l1r = (np.ascontiguousarray(win_l1)).reshape(-1, win_l1.shape[-2] * win_l1.shape[-1])

    # Linear
    preln1 = win_l1r @ W1.reshape(-1, W1.shape[-1])
    preln1 = preln1.reshape(BATCH_SIZE, X.shape[1], X.shape[1], NUM_FEATURES_L1)

    # Layer Norm
    lnmean1 = np.mean(preln1, axis = -1, keepdims = True)
    lnvar1 = np.var(preln1, axis = -1, keepdims = True)
    lnraw1 = (preln1 - lnmean1)/np.sqrt(lnvar1 + 1e-5)
    ln1 = g1 * lnraw1 + b1

    # ReLu
    rl1 = np.maximum(LEAKY_RELU_CONSTANT * ln1, ln1)

    # Max Pooling
    rl1t = rl1.transpose(0, 3, 1, 2)
    img_pieced1 = rl1t.reshape(BATCH_SIZE, NUM_FEATURES_L1, rl1t.shape[-1] // MAX_POOLING_SHRINK_L1, MAX_POOLING_SHRINK_L1, rl1t.shape[-1] // MAX_POOLING_SHRINK_L1, MAX_POOLING_SHRINK_L1)
    pooled_img1 = np.max(img_pieced1, axis=(-1, -3))

    ## Layer 2
    # Padding
    pad2 = tuple((size-1)//2 for size in WIN_SIZE_L2)
    img_padded2 = np.pad(pooled_img1, ((0,0), (0,0), pad2, pad2), "constant", constant_values=0)

    # Conv
    win_l2 = lst.sliding_window_view(img_padded2, window_shape = WIN_SIZE_L2, axis = (2,3))
    win_l2c = np.ascontiguousarray(win_l2).transpose(0, 2, 3, 4, 5, 1)
    win_l2r = win_l2c.reshape(-1, win_l2c.shape[-2] * win_l2c.shape[-1] * win_l2c.shape[-3])

    # Linear
    preln2 = win_l2r @ W2.reshape(-1, W2.shape[-1])
    preln2 = preln2.reshape(BATCH_SIZE, pooled_img1.shape[2], pooled_img1.shape[2], NUM_FEATURES_L2)

    # Layer Norm
    lnmean2 = np.mean(preln2, axis = -1, keepdims = True)
    lnvar2 = np.var(preln2, axis = -1, keepdims = True)
    lnraw2 = (preln2 - lnmean2)/np.sqrt(lnvar2 + 1e-5)
    ln2 = g2 * lnraw2 + b2

    # ReLu
    rl2 = np.maximum(LEAKY_RELU_CONSTANT * ln2, ln2)

    # Max Pooling
    rl2t = rl2.transpose(0, 2, 3, 1)
    img_pieced2 = rl2t.reshape(BATCH_SIZE, NUM_FEATURES_L2, rl2t.shape[-1] // MAX_POOLING_SHRINK_L2, MAX_POOLING_SHRINK_L2, rl2t.shape[-1] // MAX_POOLING_SHRINK_L2, MAX_POOLING_SHRINK_L2)
    pooled_img2 = np.max(img_pieced2, axis=(-1, -3))

    ## Layer 3
    # Padding
    pad3 = tuple((size-1)//2 for size in WIN_SIZE_L3)
    img_padded3 = np.pad(pooled_img2, ((0,0), (0,0), pad3, pad3), "constant", constant_values=0)

    # Conv
    win_l3 = lst.sliding_window_view(img_padded3, window_shape = WIN_SIZE_L3, axis = (2,3))
    win_l3c = np.ascontiguousarray(win_l3).transpose(0, 2, 3, 4, 5, 1)
    win_l3r = win_l3c.reshape(-1, win_l3c.shape[-2] * win_l3c.shape[-1] * win_l3c.shape[-3])

    # Linear
    preln3 = win_l3r @ W3.reshape(-1, W3.shape[-1])
    preln3 = preln3.reshape(BATCH_SIZE, pooled_img2.shape[2], pooled_img2.shape[2], NUM_FEATURES_L3)

    # Layer Norm
    lnmean3 = np.mean(preln3, axis = -1, keepdims = True)
    lnvar3 = np.var(preln3, axis = -1, keepdims = True)
    lnraw3 = (preln3 - lnmean3)/np.sqrt(lnvar3 + 1e-5)
    ln3 = g3 * lnraw3 + b3

    # ReLu
    rl3 = np.maximum(LEAKY_RELU_CONSTANT * ln3, ln3)

    # Max Pooling
    rl3t = rl3.transpose(0, 3, 1, 2)
    img_pieced3 = rl3t.reshape(BATCH_SIZE, NUM_FEATURES_L3, rl3t.shape[-1] // MAX_POOLING_SHRINK_L3, MAX_POOLING_SHRINK_L3, rl3t.shape[-1] // MAX_POOLING_SHRINK_L3, MAX_POOLING_SHRINK_L3)
    pooled_img3 = np.max(img_pieced3, axis=(-1, -3))

    ## Layer 4
    img_flattened = pooled_img3.reshape(BATCH_SIZE, -1)
    h4 = img_flattened @ W4
    mlp_data = h4.copy()

    # ReLu
    rl4 = np.maximum(LEAKY_RELU_CONSTANT * h4, h4)

    ## Layer 5
    raw_pred = rl4 @ W5

    ## Sigmoid + Loss
    safe_raw_pred = np.clip(raw_pred, -250, 250)
    probs = 1/(1 + (np.e ** -(safe_raw_pred)))
    probs = np.clip(probs, 1e-7, 1 - 1e-7)
    loss = -(Y * np.log(probs + 1e-5) + (1 - Y) * np.log(1 - probs + 1e-5))

    ### Backward Pass

    ## Sigmoid + Loss
    draw_pred = (probs - Y) / BATCH_SIZE

    ## Layer 5
    dW5 = rl4.T @ draw_pred
    drl4 = draw_pred @ W5.T

    ## Layer 4
    dh4 = np.where((rl4 >= 0), drl4, drl4 * 0.01)
    dimg_flattened = dh4 @ W4.T
    dW4 = img_flattened.T @ dh4
    dpooled_img3 = dimg_flattened.reshape(pooled_img3.shape)

    ## Layer 3
    # Max Pooling
    pooled_img3_expanded = np.max(img_pieced3, axis=(-1, -3), keepdims=True)
    # cords = np.where(np.isin(img_pieced3, pooled_img3))
    # dimg_pieced3 = np.zeros_like(img_pieced3) ## FIX DIS DERIVATIVE!!!
    mask3 = (img_pieced3 == pooled_img3_expanded) / (img_pieced3 == pooled_img3_expanded).sum(axis=(-1,-3), keepdims=True)
    dimg_pieced3 = mask3 * np.expand_dims(dpooled_img3, axis=(-1, -3))

    # if np.round(mask3.mean(), decimals=2) != 0.25:
      # print(f"WARNING 1: The mean of img_pieced3 == pooled_img3_expanded is NOT 0.25. It is {mask3.mean()}")

    # print(drl3t.shape)
    drl3t = dimg_pieced3.reshape(rl3t.shape)
    drl3 = drl3t.transpose(0, 2, 3, 1)

    # ReLu
    dln3 = np.where((ln3 >= 0), drl3, drl3 * 0.01)
    dg3 = np.sum(lnraw3 * dln3, axis = (0, 1, 2))
    dlnraw3 = dln3 * g3
    db3 = np.sum(dln3, axis = (0, 1, 2))

    # LayerNorm
    dpreln3 = (dlnraw3 - dlnraw3.mean(axis=-1, keepdims=True) - lnraw3 * (dlnraw3 * lnraw3).mean(axis=-1, keepdims=True))/(np.sqrt(lnvar3 + 1e-5))

    # Linear
    dwin_l3r = dpreln3.reshape(-1, NUM_FEATURES_L3) @ W3.reshape(-1, NUM_FEATURES_L3).T
    dW3 = (win_l3r.T @ dpreln3.reshape(-1, NUM_FEATURES_L3)).reshape(W3.shape)

    # Conv
    dwin_l3c = dwin_l3r.reshape(win_l3c.shape)
    dwin_l3 = dwin_l3c.transpose(0, 5, 1, 2, 3, 4)
    dimg_padded3 = np.zeros_like(img_padded3)
    row,col = WIN_SIZE_L3
    for r in range(row):
      for c in range(col):
        dimg_padded3[:, :, r:win_l3.shape[2]+r, c:win_l3.shape[3]+c] += dwin_l3[:, :, :, :, r, c]


    # Padding
    padw3 = int((WIN_SIZE_L3[0] - 1)//2)
    padh3 = int((WIN_SIZE_L3[1] - 1)//2)
    dpooled_img2 = dimg_padded3[:, :, padw3:-padw3, padh3:-padh3]


    ## Layer 2
    # Max Pooling
    pooled_img2_expanded = np.max(img_pieced2, axis=(-1, -3), keepdims=True)
    mask2 = (img_pieced2 == pooled_img2_expanded) / (img_pieced2 == pooled_img2_expanded).sum(axis=(-1,-3), keepdims=True)
    dimg_pieced2 = mask2 * np.expand_dims(dpooled_img2, axis=(-1, -3))

    # if np.round(mask2.mean(), decimals=2) != 1/16:
    #   print(f"WARNING 2: The mean of img_pieced2 == pooled_img2_expanded is NOT {1/16}. It is {mask2.mean()}")

    drl2t = dimg_pieced2.reshape(rl2t.shape)
    drl2 = drl2t.transpose(0, 2, 3, 1)

    # ReLu
    dln2 = np.where((ln2 >= 0), drl2, drl2 * 0.01)
    dg2 = np.sum(lnraw2 * dln2, axis = (0, 1, 2))
    dlnraw2 = dln2 * g2
    db2 = np.sum(dln2, axis = (0, 1, 2))

    # LayerNorm
    dpreln2 = (dlnraw2 - dlnraw2.mean(axis=-1, keepdims=True) - lnraw2 * (dlnraw2 * lnraw2).mean(axis=-1, keepdims=True))/(np.sqrt(lnvar2 + 1e-5))

    # Linear
    dwin_l2r = dpreln2.reshape(-1, NUM_FEATURES_L2) @ W2.reshape(-1, NUM_FEATURES_L2).T
    dW2 = (win_l2r.T @ dpreln2.reshape(-1, NUM_FEATURES_L2)).reshape(W2.shape)

    # Conv
    dwin_l2c = dwin_l2r.reshape(win_l2c.shape)
    dwin_l2 = dwin_l2c.transpose(0, 5, 1, 2, 3, 4)
    dimg_padded2 = np.zeros_like(img_padded2)

    row,col = WIN_SIZE_L2 ## MUST FIGURE OUT HOW TO REMOVE FOR LOOP W/ np.add.at
    for r in range(row):
      for c in range(col):
        dimg_padded2[:, :, r:win_l2.shape[2]+r, c:win_l2.shape[3]+c] += dwin_l2[:, :, :, :, r, c]


    # Padding
    padw2 = int((WIN_SIZE_L2[0] - 1)/2)
    padh2 = int((WIN_SIZE_L2[1] - 1)/2)
    dpooled_img1 = dimg_padded2[:, :, padw2:-padw2, padh2:-padh2]


    ## Layer 1
    # Max Pooling
    pooled_img1_expanded = np.max(img_pieced1, axis=(-1, -3), keepdims=True)
    mask1 = (img_pieced1 == pooled_img1_expanded) / (img_pieced1 == pooled_img1_expanded).sum(axis=(-1,-3), keepdims=True)
    dimg_pieced1 = mask1 * np.expand_dims(dpooled_img1, axis=(-1, -3))

    # if np.round(mask1.mean(), decimals=2) != 1/16:
    #   print(f"WARNING 3: The mean of img_pieced1 == pooled_img1_expanded is NOT {1/16}. It is {mask1.mean()}")

    drl1t = dimg_pieced1.reshape(rl1t.shape)
    drl1 = drl1t.transpose(0, 2, 3, 1)

    # ReLu
    dln1 = np.where((ln1 >= 0), drl1, drl1 * 0.01)
    dg1 = np.sum(lnraw1 * dln1, axis = (0, 1, 2))
    dlnraw1 = dln1 * g1
    db1 = np.sum(dln1, axis = (0, 1, 2))

    # LayerNorm
    dpreln1 = (dlnraw1 - dlnraw1.mean(axis=-1, keepdims=True) - lnraw1 * (dlnraw1 * lnraw1).mean(axis=-1, keepdims=True))/(np.sqrt(lnvar1 + 1e-5))

    # Linear
    dwin_l1r = dpreln1.reshape(-1, NUM_FEATURES_L1) @ W1.reshape(-1, NUM_FEATURES_L1).T
    dW1 = (win_l1r.T @ dpreln1.reshape(-1, NUM_FEATURES_L1)).reshape(W1.shape)

    # Conv
    dwin_l1 = dwin_l1r.reshape(win_l1.shape)
    dimg_padded1 = np.zeros_like(img_padded1)

    row,col = WIN_SIZE_L1
    for r in range(row):
      for c in range(col):
        dimg_padded1[:, r:win_l1.shape[1]+r, c:win_l1.shape[2]+c] += dwin_l1[:, :, :, r, c]

    # Padding
    padw1 = int((WIN_SIZE_L1[0] - 1)/2)
    padh1 = int((WIN_SIZE_L1[1] - 1)/2)
    dpooled_img1 = dimg_padded1[:, padw1:-padw1, padh1:-padh1]

    gradients = [dW1, dg1, db1, dW2, dg2, db2, dW3, dg3, db3, dW4, dW5]


    ### Update PARAPARAMETERS!!!
    lr = 0.1 if epoch < 6 else 0.01
    for parameter, gradient in list(zip(parameters, gradients)):
      update = lr * gradient
      parameter -= update

    ### Tracking Stats
    if epoch == 0 and i == 0:
      print(f"Initial Loss: {loss.mean()}")

  print(f"Epoch {epoch + 1}: Loss {loss.mean()}")


Initial Loss: 0.6964428063705079
Epoch 1: Loss 0.6964428063705079
Epoch 2: Loss 0.6919863635298338
Epoch 3: Loss 0.6728416370371125
Epoch 4: Loss 0.6303344326839512
Epoch 5: Loss 0.6193761241552586
Epoch 6: Loss 0.6029514306604625
Epoch 7: Loss 0.5933697140697087
Epoch 8: Loss 0.5863314571736152
Epoch 9: Loss 0.5820099838322195
Epoch 10: Loss 0.5773781338189091
Epoch 11: Loss 0.5708541471692303
Epoch 12: Loss 0.5633136795390644
Epoch 13: Loss 0.5543760048408775
Epoch 14: Loss 0.5509368500668383
Epoch 15: Loss 0.5480887214926674
Epoch 16: Loss 0.5402505977482118
Epoch 17: Loss 0.531958180037533
Epoch 18: Loss 0.5191399125876908
Epoch 19: Loss 0.5104616289340854
Epoch 20: Loss 0.5134303177921534


In [ ]:
mlp_data.shape
mlp_data[1]

array([-1.39628841, -1.87366849, -0.65567344,  0.52647352, -0.84726929,
        1.56696535, -1.6365904 , -0.93867669])

In [ ]:
rl3t.shape

(6, 128, 16, 16)

In [ ]:
X.shape

(6, 256, 256)

In [ ]:
## Sampling
test_img_path = f"groceries.jpeg"
img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
if img is not None:
    resized_img = cv2.resize(img, (START_SIZE, START_SIZE))
    test_img = resized_img.reshape(1, START_SIZE, START_SIZE)
    ### Forward Pass


    ## Layer 1
    # Padding
    pad1 = tuple((size-1)//2 for size in WIN_SIZE_L1) # Same Padding Calcs
    img_padded1 = np.pad(test_img, ((0,0), pad1, pad1), "constant", constant_values=0)

    # Conv
    win_l1 = lst.sliding_window_view(img_padded1, window_shape = WIN_SIZE_L1, axis = (1,2))
    win_l1r = (np.ascontiguousarray(win_l1)).reshape(-1, win_l1.shape[-2] * win_l1.shape[-1])

    # Linear
    preln1 = win_l1r @ W1.reshape(-1, W1.shape[-1])
    preln1 = preln1.reshape(1, X.shape[1], X.shape[1], NUM_FEATURES_L1)

    # Layer Norm
    lnraw1 = (preln1 - running_mean1)/np.sqrt(running_var1 + 1e-5)
    ln1 = g1 * lnraw1 + b1

    # ReLu
    rl1 = np.maximum(LEAKY_RELU_CONSTANT * ln1, ln1)

    # Max Pooling
    rl1t = rl1.transpose(0, 3, 1, 2)
    img_pieced1 = rl1t.reshape(BATCH_SIZE, NUM_FEATURES_L1, rl1t.shape[-1] // MAX_POOLING_SHRINK_L1, MAX_POOLING_SHRINK_L1, rl1t.shape[-1] // MAX_POOLING_SHRINK_L1, MAX_POOLING_SHRINK_L1)
    pooled_img1 = np.max(img_pieced1, axis=(-1, -3))


    ## Layer 2
    # Padding
    pad2 = tuple((size-1)//2 for size in WIN_SIZE_L2)
    img_padded2 = np.pad(pooled_img1, ((0,0), (0,0), pad2, pad2), "constant", constant_values=0)

    # Conv
    win_l2 = lst.sliding_window_view(img_padded2, window_shape = WIN_SIZE_L2, axis = (2,3))
    win_l2c = np.ascontiguousarray(win_l2).transpose(0, 2, 3, 4, 5, 1)
    win_l2r = win_l2c.reshape(-1, win_l2c.shape[-2] * win_l2c.shape[-1] * win_l2c.shape[-3])

    # Linear
    preln2 = win_l2r @ W2.reshape(-1, W2.shape[-1])
    preln2 = preln2.reshape(BATCH_SIZE, pooled_img1.shape[2], pooled_img1.shape[2], NUM_FEATURES_L2)

    # Layer Norm
    lnraw2 = (preln2 - running_mean2)/np.sqrt(running_var2 + 1e-5)
    ln2 = g2 * lnraw2 + b2

    # ReLu
    rl2 = np.maximum(LEAKY_RELU_CONSTANT * ln2, ln2)

    # Max Pooling
    rl2t = rl2.transpose(0, 2, 3, 1)
    img_pieced2 = rl2t.reshape(BATCH_SIZE, NUM_FEATURES_L2, rl2t.shape[-1] // MAX_POOLING_SHRINK_L2, MAX_POOLING_SHRINK_L2, rl2t.shape[-1] // MAX_POOLING_SHRINK_L2, MAX_POOLING_SHRINK_L2)
    pooled_img2 = np.max(img_pieced2, axis=(-1, -3))


    ## Layer 3
    # Padding
    pad3 = tuple((size-1)//2 for size in WIN_SIZE_L3)
    img_padded3 = np.pad(pooled_img2, ((0,0), (0,0), pad3, pad3), "constant", constant_values=0)

    # Conv
    win_l3 = lst.sliding_window_view(img_padded3, window_shape = WIN_SIZE_L3, axis = (2,3))
    win_l3c = np.ascontiguousarray(win_l3).transpose(0, 2, 3, 4, 5, 1)
    win_l3r = win_l3c.reshape(-1, win_l3c.shape[-2] * win_l3c.shape[-1] * win_l3c.shape[-3])

    # Linear
    preln3 = win_l3r @ W3.reshape(-1, W3.shape[-1])
    preln3 = preln3.reshape(BATCH_SIZE, pooled_img2.shape[2], pooled_img2.shape[2], NUM_FEATURES_L3)

    # Layer Norm
    lnraw3 = (preln3 - running_mean3)/np.sqrt(running_var3 + 1e-5)
    ln3 = g3 * lnraw3 + b3

    # ReLu
    rl3 = np.maximum(LEAKY_RELU_CONSTANT * ln3, ln3)

    # Max Pooling
    rl3t = rl3.transpose(0, 3, 1, 2)
    img_pieced3 = rl3t.reshape(BATCH_SIZE, NUM_FEATURES_L3, rl3t.shape[-1] // MAX_POOLING_SHRINK_L3, MAX_POOLING_SHRINK_L3, rl3t.shape[-1] // MAX_POOLING_SHRINK_L3, MAX_POOLING_SHRINK_L3)
    pooled_img3 = np.max(img_pieced3, axis=(-1, -3))


    ## Layer 4
    img_flattened = pooled_img3.reshape(BATCH_SIZE, -1)
    h4 = img_flattened @ W4
    mlp_data = h4.copy()

    # ReLu
    rl4 = np.maximum(LEAKY_RELU_CONSTANT * h4, h4)

    ## Layer 5
    raw_pred = rl4 @ W5

    ## Sigmoid
    probs = 1/(1 + (np.e ** -(raw_pred)))
    print(probs)

[[1.]
 [1.]
 [1.]
 [1.]
 [0.]
 [1.]]


/tmp/ipykernel_678/2702884149.py:102: RuntimeWarning: overflow encountered in power
  probs = 1/(1 + (np.e ** -(raw_pred)))
